# **04. Gán nhãn Cảm xúc & Chuẩn hóa Khía cạnh (Sentiment & Aspect Labeling)**

## **Mục tiêu:**
- Tải dữ liệu đã qua gán nhãn bằng LLM.
- Rà soát, sửa lỗi chính tả ở phần aspect.
- Kiểm tra thống kê phân bổ cảm xúc và khía cạnh.
- Chuẩn bị tập dữ liệu huấn luyện tối ưu.

In [ ]:
import os
import sys
import pandas as pd
from collections import Counter

sys.path.append(os.path.abspath('../src'))
import utils
from labeling import correct_aspect_typos, load_classified_dataset

### **1. Nạp dữ liệu kết quả gán nhãn**
Đọc file CSV chứa các nhãn cảm xúc (`LLM_Sentiment`) và khía cạnh (`LLM_Aspect`).

In [ ]:
classified_path = "../data/raw/classified_reviews.csv"

if not os.path.exists(classified_path):
    print(f"❌ File gán nhãn không tồn tại: {classified_path}")
else:
    df = load_classified_dataset(classified_path)
    print(f"✅ Đã tải và tiền xử lý {len(df):,} dòng dữ liệu đã gán nhãn.")
    display(df.head(3))

### **2. Thống kê Phân bổ Cảm xúc (Sentiment)**
Ánh xạ mã số sang chữ tiếng Việt để kiểm tra.

In [ ]:
sentiment_counts = df["LLM_Sentiment"].value_counts()
sentiment_map = {0: "0 (Tiêu cực)", 1: "1 (Trung lập)", 2: "2 (Tích cực)"}

print("📊 THỐNG KÊ PHÂN PHỐI LỚP CẢM XÚC:")
for label, count in sentiment_counts.items():
    label_name = sentiment_map.get(label, str(label))
    print(f"   + Lớp {label_name:<15}: {count:>7,} dòng ({count/len(df)*100:.2f}%)")

### **3. Thống kê Phân bổ Khía cạnh (Aspect)**
Bóc tách các khía cạnh đơn lẻ từ chuỗi đa khía cạnh (ngăn cách bởi dấu phẩy).

In [ ]:
all_aspects = []
for aspect_cell in df["LLM_Aspect"].dropna():
    parts = [a.strip() for a in str(aspect_cell).split(",") if a.strip()]
    all_aspects.extend(parts)

aspect_counts = Counter(all_aspects)
print("📊 THỐNG KÊ SỐ LƯỢT XUẤT HIỆN KHÍA CẠNH:")
for aspect, count in aspect_counts.most_common():
    print(f"   + Khía cạnh {aspect:<20}: {count:>7,} lượt ({count/len(df)*100:.2f}%)")

### **4. Xuất tập dữ liệu phục vụ Huấn luyện**
Lưu tập dữ liệu đã gán nhãn và làm sạch chính tả aspect sang `data/processed/train_dataset.csv`.

In [ ]:
df.to_csv("../data/processed/train_dataset.csv", index=False, encoding="utf-8-sig")
print("✅ Lưu tập huấn luyện thành công tại ../data/processed/train_dataset.csv")